In [1]:
import sys
from pathlib import Path

# Thêm src vào path để import module
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
from datetime import datetime

from src.optimizer import min_variance_portfolio
from src.portfolio_metrics import portfolio_stats
from src.features import build_returns_matrix

print("✅ Imports OK")
print(f"Notebook run at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ Imports OK
Notebook run at: 2026-08-21 11:04:37


In [10]:
# Chạy trước để update DB
from src.data_loader import update_db
update_db()  # sẽ mất 3-5 phút

✅ Table Stock_Prices ready
📥 Fetching 29 tickers | 2021-01-01 → 2026-08-21
   Source: vnstock KBS (fallback: yfinance)



  ✅ ACB   →    58 rows
  ✅ BID   →    58 rows
  ✅ CTG   →    58 rows
  ✅ DGC   →    58 rows
  ✅ FPT   →    58 rows
  ✅ GAS   →    58 rows
  ✅ GVR   →    58 rows
  ✅ HDB   →    58 rows
  ✅ HPG   →    58 rows
  ✅ LPB   →    58 rows
  ✅ MBB   →    58 rows
  ✅ MSN   →    58 rows
  ✅ MWG   →    58 rows
  ✅ PLX   →    58 rows
  ✅ SAB   →    58 rows
  ✅ SHB   →    58 rows
  ✅ SSB   →    58 rows
  ✅ SSI   →    58 rows
  ✅ STB   →    58 rows
  ✅ TCB   →    58 rows
  ✅ TPB   →    58 rows
  ✅ VCB   →    58 rows
  ✅ VHM   →    58 rows
  ✅ VIB   →    58 rows
  ✅ VIC   →    58 rows
  ✅ VJC   →    58 rows
  ✅ VNM   →    58 rows
  ✅ VPB   →    58 rows
  ✅ VRE   →    58 rows

✅ Done. Total new rows: 1,682

📊 DB Summary:
---------------------------------------------
No. Tickers : 29
No. Rows    : 40,661
---------------------------------------------
Ticker  rows start_date   end_date
   ACB  1404 2021-01-04 2026-08-21
   BID  1404 2021-01-04 2026-08-21
   CTG  1404 2021-01-04 2026-08-21
   DGC  1404 2021

In [16]:
BENCHMARKS = {
    "VCB + BID (high correlation)": {
        "tickers": ["VCB", "BID"],
        "description": "2 mã ngân hàng tương quan cao",
    },
    "5 diverse sectors": {
        "tickers": ["VCB", "VNM", "HPG", "FPT", "MWG"],
        "description": "Ngân hàng, tiêu dùng, thép, công nghệ, bán lẻ",
    },
    "5 banks (same sector)": {
        "tickers": ["VCB", "BID", "CTG", "MBB", "ACB"],
        "description": "5 ngân hàng lớn — tương quan trong ngành cao",
    },
    "10 VN30 stocks": {
        "tickers": ["VCB", "BID", "CTG", "TCB", "MBB", "FPT", "VNM", "HPG", "GAS", "VIC"],
        "description": "10 mã đa ngành",
    },
    "29 VN30 (full)": {
        "tickers": ["ACB","BID","CTG","DGC","FPT","GAS","GVR","HDB","HPG",
                    "LPB","MBB","MSN","MWG","PLX","SAB","SHB","SSB","SSI",
                    "STB","TCB","TPB","VCB","VHM","VIB","VIC","VJC","VNM",
                    "VPB","VRE"],
        "description": "Toàn bộ 29 mã VN30 (đã loại VPL do mới niêm yết)",
    },
}

START_DATE = "2021-01-01"
END_DATE = "2026-08-21"
RF = 0.045  # SBV operating rate

print(f"Sẽ chạy {len(BENCHMARKS)} portfolio benchmarks")
print(f"Data range: {START_DATE} → {END_DATE}")
print(f"Risk-free rate: {RF*100}%")

Sẽ chạy 5 portfolio benchmarks
Data range: 2021-01-01 → 2026-08-21
Risk-free rate: 4.5%


In [17]:
def run_benchmark(name, tickers, start, end, rf=0.045):
    """Chạy MVP optimization + tính stats cho 1 tổ hợp."""
    print(f"\n{'='*60}")
    print(f"Benchmark: {name}")
    print(f"Tickers ({len(tickers)}): {', '.join(tickers)}")
    print(f"{'='*60}")
    
    # Chạy MVP
    result = min_variance_portfolio(tickers, start, end)
    
    if not result.get('success', False):
        print(f"⚠️  Optimizer không hội tụ cho {name}")
        return None
    
    weights = np.array(result['weights'])
    cov = result['cov']
    mu = result['mu']
    n = len(weights)
    
    # MVP stats
    mvp_return = float(weights @ mu.values if hasattr(mu, 'values') else mu)
    mvp_vol = float(np.sqrt(weights @ cov @ weights))
    mvp_sharpe = (mvp_return - rf) / mvp_vol if mvp_vol > 0 else 0
    
    # Equal Weights baseline
    ew = np.ones(n) / n
    ew_return = float(ew @ (mu.values if hasattr(mu, 'values') else mu))
    ew_vol = float(np.sqrt(ew @ cov @ ew))
    ew_sharpe = (ew_return - rf) / ew_vol if ew_vol > 0 else 0
    
    # Vol reduction
    vol_reduction_pct = (ew_vol - mvp_vol) / ew_vol * 100
    
    output = {
        "portfolio": name,
        "n_assets": n,
        "mvp_return": mvp_return,
        "mvp_vol": mvp_vol,
        "mvp_sharpe": mvp_sharpe,
        "ew_return": ew_return,
        "ew_vol": ew_vol,
        "ew_sharpe": ew_sharpe,
        "vol_reduction_pct": vol_reduction_pct,
    }
    
    # In kết quả
    print(f"  MVP Return: {mvp_return*100:.2f}%")
    print(f"  MVP Vol:    {mvp_vol*100:.2f}%")
    print(f"  MVP Sharpe: {mvp_sharpe:.3f}")
    print(f"  EW Return:  {ew_return*100:.2f}%")
    print(f"  EW Vol:     {ew_vol*100:.2f}%")
    print(f"  Vol Reduction: {vol_reduction_pct:.1f}%")
    
    return output

In [18]:
results = []
for name, config in BENCHMARKS.items():
    result = run_benchmark(name, config["tickers"], START_DATE, END_DATE, RF)
    if result:
        results.append(result)

df = pd.DataFrame(results)
df


Benchmark: VCB + BID (high correlation)
Tickers (2): VCB, BID
---------------------------------------------
No. Tickers : 29
No. Rows    : 40,661
---------------------------------------------
  MVP Return: 8.98%
  MVP Vol:    24.51%
  MVP Sharpe: 0.183
  EW Return:  9.30%
  EW Vol:     25.47%
  Vol Reduction: 3.8%

Benchmark: 5 diverse sectors
Tickers (5): VCB, VNM, HPG, FPT, MWG
---------------------------------------------
No. Tickers : 29
No. Rows    : 40,661
---------------------------------------------
  MVP Return: 7.10%
  MVP Vol:    19.64%
  MVP Sharpe: 0.132
  EW Return:  11.49%
  EW Vol:     21.46%
  Vol Reduction: 8.5%

Benchmark: 5 banks (same sector)
Tickers (5): VCB, BID, CTG, MBB, ACB
---------------------------------------------
No. Tickers : 29
No. Rows    : 40,661
---------------------------------------------
  MVP Return: 12.06%
  MVP Vol:    22.65%
  MVP Sharpe: 0.334
  EW Return:  14.80%
  EW Vol:     24.84%
  Vol Reduction: 8.8%

Benchmark: 10 VN30 stocks
Tickers

,portfolio,n_assets,mvp_return,mvp_vol,mvp_sharpe,ew_return,ew_vol,ew_sharpe,vol_reduction_pct
0,VCB + BID (high correlation),2,0.089804,0.245076,0.182817,0.093047,0.254689,0.188651,3.774737
1,5 diverse sectors,5,0.070997,0.196437,0.132344,0.114929,0.214590,0.325872,8.459360
2,5 banks (same sector),5,0.120622,0.226524,0.333837,0.147991,0.248381,0.414651,8.799883
3,10 VN30 stocks,10,0.103889,0.186610,0.315572,0.151444,0.212484,0.500954,12.176564
4,29 VN30 (full),29,0.072538,0.156231,0.176267,0.143425,0.210773,0.466969,25.876966


In [14]:
# Lưu để tham khảo
df.to_csv("../reports/benchmark_results_20260821.csv", index=False)
df.to_json("../reports/benchmark_results_20260821.json", orient="records", indent=2)

print("✅ Đã lưu kết quả benchmark")
print(f"CSV: reports/benchmark_results_20260821.csv")
print(f"JSON: reports/benchmark_results_20260821.json")

✅ Đã lưu kết quả benchmark
CSV: reports/benchmark_results_20260821.csv
JSON: reports/benchmark_results_20260821.json


In [15]:
def format_markdown_table(df):
    lines = []
    lines.append("| Portfolio | # Assets | MVP Return | MVP Vol | EW Vol | Vol Reduction | Sharpe (MVP) |")
    lines.append("|-----------|----------|------------|---------|--------|---------------|--------------|")
    
    for _, row in df.iterrows():
        # Bold row cho full VN30
        is_full = "29 VN30" in row["portfolio"]
        wrap = "**" if is_full else ""
        
        line = (
            f"| {wrap}{row['portfolio']}{wrap} "
            f"| {wrap}{row['n_assets']}{wrap} "
            f"| {wrap}{row['mvp_return']*100:.2f}%{wrap} "
            f"| {wrap}{row['mvp_vol']*100:.2f}%{wrap} "
            f"| {wrap}{row['ew_vol']*100:.2f}%{wrap} "
            f"| {wrap}{row['vol_reduction_pct']:.1f}%{wrap} "
            f"| {wrap}{row['mvp_sharpe']:.3f}{wrap} |"
        )
        lines.append(line)
    
    return "\n".join(lines)

markdown = format_markdown_table(df)
print(markdown)

| Portfolio | # Assets | MVP Return | MVP Vol | EW Vol | Vol Reduction | Sharpe (MVP) |
|-----------|----------|------------|---------|--------|---------------|--------------|
| VCB + BID (high correlation) | 2 | 8.98% | 24.51% | 25.47% | 3.8% | 0.183 |
| 5 diverse sectors | 5 | 7.10% | 19.64% | 21.46% | 8.5% | 0.132 |
| 5 banks (same sector) | 5 | 12.06% | 22.65% | 24.84% | 8.8% | 0.334 |
| 10 VN30 stocks | 10 | 10.39% | 18.66% | 21.25% | 12.2% | 0.316 |
| **29 VN30 (full)** | **29** | **7.25%** | **15.62%** | **21.08%** | **25.9%** | **0.176** |
